In [ ]:
# install gdown
!pip install gdown

In [ ]:
#importing below libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gdown

In [ ]:
#importing instagram_usage_lifestyle data
file_id = "16nG7VaTdxL1xpi3ZuYrUf5lTV7bvTyxP"
download_url = f"https://docs.google.com/uc?export=download&id={file_id}"
output_filename = "instagram_usage_lifestyle.csv"
gdown.download(download_url, output_filename, quiet=False)
instagram_df = pd.read_csv("instagram_usage_lifestyle.csv")
instagram_df.head()

Downloading...
From (original): https://docs.google.com/uc?export=download&id=16nG7VaTdxL1xpi3ZuYrUf5lTV7bvTyxP
From (redirected): https://docs.google.com/uc?export=download&id=16nG7VaTdxL1xpi3ZuYrUf5lTV7bvTyxP&confirm=t&uuid=280aa389-ff5a-442e-bb94-1dfad57e920e
To: /content/instagram_usage_lifestyle.csv
100%|██████████| 440M/440M [00:07<00:00, 60.0MB/s]


,user_id,app_name,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,...,last_login_date,average_session_length_minutes,content_type_preference,preferred_content_theme,privacy_setting_level,two_factor_auth_enabled,biometric_login_used,linked_accounts_count,subscription_status,user_engagement_score
0,1,Instagram,51,Female,India,Rural,High,Retired,Bachelor’s,Single,...,2025-11-02,5.0,Mixed,Tech,Private,Yes,No,0,Free,7.83
1,2,Instagram,64,Female,United Kingdom,Urban,Middle,Full-time employed,Other,Divorced,...,2025-03-22,14.8,Photos,Fashion,Public,No,No,3,Free,1.43
2,3,Instagram,41,Female,Canada,Urban,Middle,Student,Bachelor’s,In a relationship,...,2025-08-10,5.0,Mixed,Other,Public,Yes,Yes,1,Free,9.67
3,4,Instagram,27,Non-binary,South Korea,Urban,Middle,Unemployed,Master’s,In a relationship,...,2025-03-31,25.9,Stories,Tech,Private,No,No,1,Free,0.94
4,5,Instagram,55,Male,India,Urban,Upper-middle,Full-time employed,Bachelor’s,Single,...,2025-03-19,13.1,Videos,Food,Public,Yes,No,0,Free,1.03


In [ ]:
print(instagram_df.shape)

(1547896, 58)


**Finding Duplicate Values**

In [ ]:
instagram_df.duplicated().sum()

np.int64(0)

**Finding Null Values**

In [ ]:
instagram_df.isnull().sum().sum()

np.int64(0)

**Removing the unwanted columns**

In [ ]:
remove_columns = [

    'age',
    'country',
    'gender',
    'income_level',
    'education_level',
    'relationship',
    'has_children',
    'exercise_hours',
    'diet_quality',
    'user_id',
    'app_name',
    'smoking',
    'alcohol_frequency',
    'body_mass_index',
    'blood_pressure_systolic',
    'blood_pressure_diastolic',
    'books_read_per_year',
    'volunteer_hours_per_month',
    'travel_frequency_per_year',
    'user_engagement_score'
]

**Finding Key Metrics**

In [ ]:
key_metric_columns = ['likes_given_per_day','comments_written_per_day', 'dms_sent_per_week', 'posts_created_per_week', 'time_on_feed_per_day', 'time_on_reels_per_day',
                'time_on_explore_per_day', 'time_on_messages_per_day', 'dms_received_per_week', 'ads_clicked_per_day', 'ads_viewed_per_day']
key_metric_columns = [m for m in key_metric_columns if m in instagram_df.columns]
print("Pimary_key_metrics: ", key_metric_columns)
print("No.of key metric columns: ",len(key_metric_columns))

Pimary_key_metrics:  ['likes_given_per_day', 'comments_written_per_day', 'dms_sent_per_week', 'posts_created_per_week', 'time_on_feed_per_day', 'time_on_reels_per_day', 'time_on_explore_per_day', 'time_on_messages_per_day', 'dms_received_per_week', 'ads_clicked_per_day', 'ads_viewed_per_day']
No.of key metric columns:  11


**Finding Outliers**

In [ ]:
outlier_summary=[]
for columns in key_metric_columns:
  q1=instagram_df[columns].quantile(0.25)
  q3=instagram_df[columns].quantile(0.75)
  iqr=q3-q1
  lower_bound=q1-1.5*iqr
  upper_bound=q3+1.5*iqr

  check= ((instagram_df[columns]<lower_bound) | (instagram_df[columns]>upper_bound))

  outlier_records = instagram_df.loc[check,['user_id', columns]]

  print(f"\noutliers count in {columns}: {len(outlier_records)}")
  print(f"outliers percentage in {columns}: {round((len(outlier_records)*100/len(instagram_df)),2)}")
  print(outlier_records.head().reset_index())



outliers count in likes_given_per_day: 409
outliers percentage in likes_given_per_day: 0.03
   index  user_id  likes_given_per_day
0    497      498                  294
1   3808     3809                  293
2   7123     7124                  300
3  11800    11801                  303
4  19178    19179                  293

outliers count in comments_written_per_day: 0
outliers percentage in comments_written_per_day: 0.0
Empty DataFrame
Columns: [index, user_id, comments_written_per_day]
Index: []

outliers count in dms_sent_per_week: 3552
outliers percentage in dms_sent_per_week: 0.23
   index  user_id  dms_sent_per_week
0    225      226                 69
1    346      347                 67
2    554      555                 65
3    556      557                 65
4   1183     1184                 68

outliers count in posts_created_per_week: 32374
outliers percentage in posts_created_per_week: 2.09
   index  user_id  posts_created_per_week
0     17       18                      1

**Capping outliers using 1st and 99th percentiles**

In [ ]:
for col in outlier_summary:

    lower_limit = instagram_df[col].quantile(0.01)

    upper_limit = instagram_df[col].quantile(0.99)

    instagram_df[col] = instagram_df[col].clip(
        lower=lower_limit,
        upper=upper_limit
    )

print("Capping completed")

Capping completed


In [ ]:
import pandas as pd

# Select required columns

selected_columns = [

    'user_id',
    'age',
    'gender',
    'urban_rural',
    'income_level',
    'employment_status',
    'exercise_hours_per_week',
    'sleep_hours_per_night',
    'diet_quality',
    'perceived_stress_score',
    'daily_active_minutes_instagram',
    'sessions_per_day',
    'posts_created_per_week',
    'reels_watched_per_day',
    'stories_viewed_per_day',
    'likes_given_per_day',
    'comments_written_per_day',
    'dms_sent_per_week',
    'dms_received_per_week',
    'ads_viewed_per_day',
    'ads_clicked_per_day',
    'time_on_feed_per_day',
    'time_on_explore_per_day',
    'time_on_messages_per_day',
    'time_on_reels_per_day',
    'uses_premium_features',
    'notification_response_rate',
    'last_login_date',
    'average_session_length_minutes',
    'content_type_preference',
    'preferred_content_theme',
    'privacy_setting_level',
    'linked_accounts_count',
    'subscription_status'

]

# Create final dataframe

final_df = instagram_df[selected_columns]

# Verify shape before export

print("Final Dataset Shape:")
print(final_df.shape)

# Expected:
# (1547896, 34)

# Export CSV

file_name = "user-bahavior.csv"

final_df.to_csv(
    file_name,
    index=False
)

print("CSV Exported Successfully")

# Verify exported file

check_df = pd.read_csv(file_name)

print("Export Verification:")
print(check_df.shape)

# Download file

from google.colab import files

files.download(file_name)

Final Dataset Shape:
(1547896, 34)
CSV Exported Successfully
Export Verification:
(1547896, 34)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>